<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day_19_EY_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Annotation — Workspace placeholder
This cell is empty and has no effect on execution. It can be removed during notebook cleanup, or retained as a space for trainer notes.

Writing .env


### Annotation — Install the RAG dependencies
This installs the libraries used across the notebook: LangChain for document and chain abstractions, text splitters for chunking, FAISS for local vector search, Azure AI Search for managed search, Azure identity helpers, and `python-dotenv` for loading configuration. `-q` reduces installation output. If package versions conflict, restart the Colab runtime after installation and rerun cells from the top.

In [2]:
!pip install -q langchain langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu azure-search-documents azure-identity python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


### Annotation — Inspect required environment settings
`os.getenv()` reads configuration variables from the current process environment. The loop checks the Azure endpoint, chat deployment, embedding deployment, and API version. These values tell the notebook where to connect and which deployed models to call. Avoid printing API keys or other secrets; this cell intentionally checks non-secret configuration only.

In [3]:
import os

for name in [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_OPENAI_EMBED_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION",
]:
    print(f"{name}: {os.getenv(name)}")

AZURE_OPENAI_ENDPOINT: None
AZURE_OPENAI_CHAT_DEPLOYMENT: None
AZURE_OPENAI_EMBED_DEPLOYMENT: None
AZURE_OPENAI_API_VERSION: None


### Annotation — Test Azure embeddings through the v1-compatible endpoint
The cell reads the endpoint and API key, initializes the OpenAI-compatible client, and requests an embedding for the sample text `hello`. An embedding is a numerical vector representing text semantics. `len(response.data[0].embedding)` reports the vector dimension; later vector indexes must use the same dimension. For production, avoid printing full endpoints and never print the API key.

In [6]:

import os
from dotenv import load_dotenv
from openai import OpenAI

# Step 1: Load variables from the .env file
env_loaded = load_dotenv("/content/.env", override=True)

print(".env file loaded:", env_loaded)

# Step 2: Read the configuration
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/")
api_key = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
embedding_deployment = os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT", "").strip()

# Step 3: Validate required settings
missing = []

if not endpoint:
    missing.append("AZURE_OPENAI_ENDPOINT")

if not api_key:
    missing.append("AZURE_OPENAI_API_KEY")

if not embedding_deployment:
    missing.append("AZURE_OPENAI_EMBED_DEPLOYMENT")

if missing:
    raise ValueError(
        "Missing .env variables: " + ", ".join(missing)
    )

# Step 4: Initialize Azure OpenAI v1 client
if endpoint.endswith("/openai/v1"):
    base_url = endpoint + "/"
elif endpoint.endswith("/openai"):
    base_url = endpoint + "/v1/"
else:
    base_url = endpoint + "/openai/v1/"

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

# Step 5: Test embeddings
response = client.embeddings.create(
    model=embedding_deployment,
    input="hello"
)

print("Embedding dimension:", len(response.data[0].embedding))
print("Azure OpenAI embedding test successful!")


.env file loaded: True
Embedding dimension: 1536
Azure OpenAI embedding test successful!


### Annotation — Create a reusable embedding initializer
`get_embeddings()` gathers the Azure settings and returns a LangChain `AzureOpenAIEmbeddings` object. Wrapping setup in a function avoids duplicating connection logic. The endpoint parsing removes the `/openai/...` API path before passing the resource endpoint to this integration. This is an earlier client configuration path; the final working v1 implementation appears later in the notebook.

In [7]:
import os
from openai import AzureOpenAI
from langchain_openai import AzureOpenAIEmbeddings

def get_embeddings():
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    api_key = os.environ["AZURE_OPENAI_API_KEY"]
    deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]
    api_version = os.environ["AZURE_OPENAI_API_VERSION"]

    return AzureOpenAIEmbeddings(
        azure_endpoint=endpoint.split("/openai/")[0],
        api_key=api_key,
        azure_deployment=deployment,
        openai_api_version=api_version,
    )

### Annotation — Verify the embedding helper
This cell calls `get_embeddings()`, embeds a short sample string, and prints the resulting vector length. A successful result confirms the embedding deployment is reachable and returns vectors. This is a smoke test, not a retrieval test: no documents or vector database are involved yet.

In [8]:
emb = get_embeddings()

vector = emb.embed_query("hello")

print("Embedding dimension:", len(vector))
print("Azure OpenAI embeddings working!")

Embedding dimension: 1536
Azure OpenAI embeddings working!


### Annotation — Repeat the embedding smoke test
This repeats the same validation as the preceding cell in a shorter form. It is useful while debugging, but redundant in a cleaned teaching notebook. Keep one embedding smoke test to reduce repeated API calls.

In [9]:
emb = get_embeddings()
print("Embedding dimension:", len(emb.embed_query("hello")))

Embedding dimension: 1536


### Annotation — Inspect chat settings before a completion call
The cell reads the chat deployment and API version, then prepares endpoint configuration for the chat test. It helps distinguish a missing or incorrect deployment setting from a model-call error. Deployment names must match the names configured in Azure AI Foundry exactly.

In [10]:
import os

print("Chat deployment:", os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"))
print("API version:", os.getenv("AZURE_OPENAI_API_VERSION"))

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
print("Endpoint:", endpoint)

Chat deployment: gpt-4o-1
API version: 2024-10-21
Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1


### Annotation — Test a chat completion independently
This sends a small prompt to the configured chat deployment and prints the returned message. The `messages` list represents the conversation; `model` identifies the deployment; and `max_tokens` limits the response length. Testing chat independently helps isolate Azure connectivity issues before introducing retrieval and prompt context.

In [11]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "user", "content": "Reply with the single word: OK"}
    ],
    max_tokens=10
)

print("Chat deployment response:", response.choices[0].message.content)

Chat deployment response: OK


### Annotation — Build the policy corpus
This cell defines the source policy records as dictionaries containing an ID, title, and policy content. The collection of dictionaries is the corpus that later stages split, embed, index, and retrieve. Stable IDs such as `HR-001` and `SEC-003` let the application trace an answer back to its source policy. The corpus content is illustrative training data, not a substitute for an organization's approved policies.

In [12]:
# Day 19 - Lab 1: Create the PolicyPilot document corpus

documents = [
    {
        "id": "HR-001",
        "title": "Work From Home Policy",
        "content": """
        Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.
        """
    },
    {
        "id": "TRV-002",
        "title": "Business Travel Policy",
        "content": """
        Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.
        """
    },
    {
        "id": "SEC-003",
        "title": "Information Security Policy",
        "content": """
        Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.
        """
    },
    {
        "id": "CDP-004",
        "title": "Learning and Development Policy",
        "content": """
        Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.
        """
    },
    {
        "id": "PRC-005",
        "title": "Procurement Policy",
        "content": """
        Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.
        """
    }
]

print("Number of policy documents:", len(documents))

for doc in documents:
    print(doc["id"], "-", doc["title"])

Number of policy documents: 5
HR-001 - Work From Home Policy
TRV-002 - Business Travel Policy
SEC-003 - Information Security Policy
CDP-004 - Learning and Development Policy
PRC-005 - Procurement Policy


### Annotation — Install the text-splitting package
This ensures `langchain-text-splitters` is installed. The package provides the recursive splitter used in the next cell. It is already included in the initial installation cell, so this separate install is optional unless the notebook is run out of order.

In [13]:
%pip install -q langchain-text-splitters

### Annotation — Split policy content into overlapping chunks
`RecursiveCharacterTextSplitter` attempts to split at paragraph breaks first, then line breaks, spaces, and finally character boundaries. `chunk_size=400` sets the target chunk length in characters; `chunk_overlap=60` repeats some text between adjacent chunks to preserve boundary context. The loops split each policy and store its source ID, title, chunk number, and text. Chunk size and overlap should be tuned to the document structure and retrieval results.

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

for doc in documents:
    doc_chunks = text_splitter.split_text(doc["content"])

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "id": doc["id"],
            "title": doc["title"],
            "chunk_number": i + 1,
            "text": chunk
        })

print("Total chunks created:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Total chunks created: 5

First chunk:
{'id': 'HR-001', 'title': 'Work From Home Policy', 'chunk_number': 1, 'text': 'Employees may work from home up to two days per week with\n        manager approval. Employees must remain available during\n        core business hours from 10 AM to 4 PM. Confidential company\n        information must only be accessed through approved systems.'}


### Annotation — Install FAISS for local vector search
This installs FAISS and the LangChain community integrations. FAISS is a local similarity-search library; unlike a hosted vector database, the index generally lives in the notebook runtime. Colab runtime resets can remove local state, so this is useful for a demonstration but not persistent cloud storage.

In [15]:
%pip install -q faiss-cpu langchain-community

### Annotation — Convert chunks into LangChain Document objects
`Document` standardizes each chunk as `page_content` plus `metadata`. The department mapping enriches each source with a department label. Metadata fields such as `source`, `title`, `department`, and `chunk_id` help with traceability, filtering, and citations. Check that every source ID in the corpus exists in `department_map` before running the conversion.

In [16]:
from langchain_core.documents import Document

department_map = {
    "HR-001": "HR",
    "TRV-002": "Finance",
    "SEC-003": "IT",
    "CDP-004": "Legal",
    "PRC-005": "Finance"
}

docs = []

for chunk in chunks:
    docs.append(
        Document(
            page_content=chunk["text"],
            metadata={
                "source": chunk["id"],
                "title": chunk["title"],
                "department": department_map[chunk["id"]],
                "chunk_id": f'{chunk["id"]}-{chunk["chunk_number"] - 1}'
            }
        )
    )

print("Documents prepared:", len(docs))
print("Sample metadata:", docs[0].metadata)

Documents prepared: 5
Sample metadata: {'source': 'HR-001', 'title': 'Work From Home Policy', 'department': 'HR', 'chunk_id': 'HR-001-0'}


### Annotation — Inspect installed package versions
`importlib.metadata.version()` reports the installed versions of `langchain-openai` and `openai`. Version checks are useful when debugging SDK differences or dependency conflicts. They do not make API calls or modify the RAG data.

In [17]:
from importlib.metadata import version

print("langchain-openai:", version("langchain-openai"))
print("openai:", version("openai"))

langchain-openai: 1.7.3
openai: 2.54.0


### Annotation — Initialize embeddings with the Azure OpenAI v1 endpoint
This configures `OpenAIEmbeddings` with the Azure deployment name, API key, and v1 base URL. The embedding model converts both documents and user questions into vectors in the same semantic space. Use the same embedding deployment consistently for indexing and querying; changing models may require re-embedding the corpus.

In [18]:
import os
from langchain_openai import OpenAIEmbeddings

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")

emb = OpenAIEmbeddings(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    base_url=endpoint + "/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

vector = emb.embed_query("Test Azure embeddings")

print("Embedding dimension:", len(vector))
print("Azure v1 embeddings working!")

Embedding dimension: 1536
Azure v1 embeddings working!


### Annotation — Build the FAISS vector store
`FAISS.from_documents(docs, emb)` embeds the prepared document chunks and creates a local vector index. `faiss_store` is the reusable retrieval object. This is the first point where the notebook combines the prepared documents and embeddings into a searchable store.

In [19]:
from langchain_community.vectorstores import FAISS

# Use the working Azure v1 embedding client
faiss_store = FAISS.from_documents(docs, emb)

print("FAISS vector database created successfully!")
print("Total policy chunks indexed:", len(docs))

/tmp/ipykernel_2575/678698563.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS vector database created successfully!
Total policy chunks indexed: 5


### Annotation — Retrieve policy chunks for a travel question
`similarity_search(query, k=3)` returns the three most similar chunks for the travel approval question. The loop prints the rank, metadata, and text so you can inspect whether retrieval found the right evidence. This checks retrieval quality only; it does not ask an LLM to generate an answer.

In [20]:
query = "What approval is required before booking business travel?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: TRV-002
Title: Business Travel Policy
Content: Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.

Result 2
Source: PRC-005
Title: Procurement Policy
Content: Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


### Annotation — Retrieve security evidence for a phishing question
This repeats the FAISS retrieval test with a security-related query. Comparing the returned source and text against the expected security policy helps validate semantic retrieval. If the wrong chunk appears, inspect the corpus, chunk boundaries, and embedding configuration before changing the generation prompt.

In [21]:
query = "What should employees do if they receive a phishing email?"

results = faiss_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print("Content:", doc.page_content)


Result 1
Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Result 2
Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Result 3
Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


### Annotation — Install the Azure AI Search SDK
This installs the Azure AI Search client library used to create indexes, upload documents, and issue searches. It is also present in the initial dependency installation, so the cell mainly makes this section easier to run independently.

In [22]:
%pip install -q azure-search-documents

### Annotation — Define the Azure AI Search index schema
This cell connects to Azure AI Search and defines the `policypilot-index` schema. Fields include a unique key, policy text, source, title, department, and a vector field. The vector field's dimensions must match the embedding model output. The vector search profile and HNSW configuration specify how Azure AI Search performs nearest-neighbour search. Index schema changes may require updating or recreating an existing index.

In [23]:
from google.colab import userdata
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SearchField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)

endpoint = userdata.get("AZURE_SEARCH_ENDPOINT")
key = userdata.get("AZURE_SEARCH_KEY")

index_name = "policypilot-index"

fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True
    ),
    SearchableField(
        name="content",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="source",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchableField(
        name="title",
        type=SearchFieldDataType.String
    ),
    SearchableField(
        name="department",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    SearchField(
        name="content_vector",
        type=SearchFieldDataType.Collection(
            SearchFieldDataType.Single
        ),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="policy-vector-profile"
    ),
]

vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(name="policy-hnsw")
    ],
    profiles=[
        VectorSearchProfile(
            name="policy-vector-profile",
            algorithm_configuration_name="policy-hnsw"
        )
    ]
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search
)

index_client = SearchIndexClient(
    endpoint=endpoint,
    credential=__import__(
        "azure.core.credentials", fromlist=["AzureKeyCredential"]
    ).AzureKeyCredential(key)
)

result = index_client.create_or_update_index(index)

print("Index created successfully:", result.name)

Index created successfully: policypilot-index


### Annotation — Inspect the created Azure AI Search index
`get_index()` retrieves the index definition from the service. The loop prints its field names and settings, allowing you to confirm the key field, searchable fields, metadata fields, vector dimensions, and vector profile. This is a configuration verification step, not a document retrieval step.

In [24]:
existing_index = index_client.get_index("policypilot-index")

print("Existing index:", existing_index.name)
print("\nExisting fields:")

for field in existing_index.fields:
    print(
        f"Name: {field.name}, "
        f"Type: {field.type}, "
        f"Key: {field.key}, "
        f"Searchable: {field.searchable}, "
        f"Filterable: {field.filterable}"
    )

Existing index: policypilot-index

Existing fields:
Name: id, Type: SearchFieldDataType.STRING, Key: True, Searchable: False, Filterable: False
Name: content, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: source, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: title, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: False
Name: department, Type: SearchFieldDataType.STRING, Key: False, Searchable: True, Filterable: True
Name: content_vector, Type: Collection(Edm.Single), Key: False, Searchable: True, Filterable: False


### Annotation — Load and validate `.env` configuration
`load_dotenv('.env', override=True)` loads variables from the file in the current working directory and allows file values to replace existing environment values. The list of required names is checked before later cells use them. Never print API-key values; report only whether each required variable is present. Ensure `.env` is excluded from Git.

In [25]:
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True)

required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_EMBED_DEPLOYMENT",
    "AZURE_OPENAI_CHAT_DEPLOYMENT",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_SEARCH_KEY",
]

missing = [name for name in required_vars if not os.getenv(name)]

if missing:
    print("Missing configuration:", missing)
    print("Please load these values from Colab Secrets.")
else:
    print("Configuration check passed.")
    print("Endpoint:", os.environ["AZURE_OPENAI_ENDPOINT"])
    print("Embedding deployment:", os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"])
    print("Chat deployment:", os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"])

Configuration check passed.
Endpoint: https://sb-mazeazusrqn9t23-dp6o-resource.openai.azure.com/openai/v1
Embedding deployment: text-embedding-3-small
Chat deployment: gpt-4o-1


### Annotation — Recheck the embedding deployment
This makes a small embedding request using the configured deployment and prints the vector length. It is a connectivity check before uploading content to Azure AI Search. The output dimension should agree with the `content_vector` field's configured dimension.

In [26]:
import os
from openai import OpenAI

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
api_key = os.environ["AZURE_OPENAI_API_KEY"]
deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]

client = OpenAI(
    api_key=api_key,
    base_url=endpoint + "/"
)

response = client.embeddings.create(
    model=deployment,
    input="Test Azure embeddings"
)

print("Embedding test successful")
print("Embedding dimension:", len(response.data[0].embedding))

Embedding test successful
Embedding dimension: 1536


### Annotation — Inspect prepared chunks and metadata
The cell prints the total number of `Document` chunks and shows the first chunk's text and metadata. This is a useful data-quality checkpoint: verify that chunk text is not empty and metadata contains the expected source, title, and department before indexing.

In [27]:
print("Number of policy chunks:", len(docs))
print("\nFirst policy chunk:")
print("Content:", docs[0].page_content)
print("Metadata:", docs[0].metadata)

Number of policy chunks: 5

First policy chunk:
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.
Metadata: {'source': 'HR-001', 'title': 'Work From Home Policy', 'department': 'HR', 'chunk_id': 'HR-001-0'}


### Annotation — Connect to Azure services and upload vectors
This cell initializes the Azure OpenAI-compatible client and an Azure AI Search `SearchClient`. It iterates through `docs`, embeds each chunk's `page_content`, and creates a search document containing its text, metadata, and vector. The upload call writes these records to the index. Review upload results for per-document failures; a successful API call alone does not guarantee every record was accepted.

In [28]:
import os
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

# 1. Connect to Azure OpenAI v1 endpoint
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=endpoint + "/"
)

embedding_deployment = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]

# 2. Connect to the existing Azure AI Search index
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

# 3. Generate embeddings and prepare search documents
search_documents = []

for i, doc in enumerate(docs):
    embedding_response = openai_client.embeddings.create(
        model=embedding_deployment,
        input=doc.page_content
    )

    vector = embedding_response.data[0].embedding
    metadata = doc.metadata

    search_documents.append({
        "id": str(i + 1),
        "content": doc.page_content,
        "source": str(metadata.get("source", "unknown")),
        "title": str(metadata.get("title", "Untitled")),
        "department": str(metadata.get("department", "General")),
        "content_vector": vector
    })

print("Prepared documents:", len(search_documents))
print("Embedding dimensions:",
      len(search_documents[0]["content_vector"]))

# 4. Upload documents to the existing index
results = search_client.upload_documents(
    documents=search_documents
)

successful = sum(1 for result in results if result.succeeded)
failed = len(results) - successful

print("Successfully uploaded:", successful)
print("Failed uploads:", failed)

for result in results:
    if not result.succeeded:
        print("Failed document:", result.key, result.error_message)

Prepared documents: 5
Embedding dimensions: 1536
Successfully uploaded: 5
Failed uploads: 0


### Annotation — Verify indexed content with text search
This query searches for the words `phishing security policy` and requests selected fields for up to three results. This is a keyword/text search, not the explicit vector query used later. It verifies that documents were uploaded and that the expected fields can be retrieved.

In [29]:
print("Documents currently in the index:")

for result in search_client.search(
    search_text="phishing security policy",
    select=["id", "title", "source", "department", "content"],
    top=3
):
    print("\nTitle:", result.get("title"))
    print("Source:", result.get("source"))
    print("Department:", result.get("department"))
    print("Content:", result.get("content"))

Documents currently in the index:

Title: Information Security Policy
Source: SEC-003
Department: IT
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Title: Work From Home Policy
Source: HR-001
Department: HR
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Title: Procurement Policy
Source: PRC-005
Department: Finance
Content: Purchases must follow the company's approval matrix.
        Employees must obtain the required approvals before
        committing company funds. Vendor selection must follow
        applicable procurement and compliance requirements.


### Annotation — Prepare clients for vector retrieval
This cell imports and initializes the clients and vector-query classes needed to run Azure AI Search vector search. It separates setup from the next cell's query execution. Reusing a single initialized client is normally more efficient than recreating it repeatedly.

In [30]:
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
import os

# Azure OpenAI client
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/"
)

# Azure AI Search client
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

print("Azure OpenAI and Azure AI Search clients are ready.")

Azure OpenAI and Azure AI Search clients are ready.


### Annotation — Run a vector query against Azure AI Search
The user question is embedded with the same embedding deployment used for documents. `VectorizedQuery` sends that vector to the `content_vector` field and requests the nearest three matches. The result loop prints the matching policy fields. Using the same model for documents and questions is essential for meaningful similarity scores.

In [31]:
query = "What should employees do if they receive a phishing email?"

# Convert the question into an embedding
embedding_response = openai_client.embeddings.create(
    model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
    input=query
)

query_vector = embedding_response.data[0].embedding

# Search using vector similarity
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=3,
    fields="content_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["id", "title", "source", "department", "content"],
    top=3
)

for rank, result in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Title:", result.get("title"))
    print("Source:", result.get("source"))
    print("Department:", result.get("department"))
    print("Content:", result.get("content"))


Result 1
Title: Information Security Policy
Source: SEC-003
Department: IT
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

Result 2
Title: Work From Home Policy
Source: HR-001
Department: HR
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Result 3
Title: Learning and Development Policy
Source: CDP-004
Department: Legal
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with their teams.


### Annotation — Test vector retrieval with multiple questions
The loop repeats embedding and vector search for two different questions. This checks whether the index can retrieve appropriate evidence across policy topics. The results should be reviewed against the actual corpus rather than assuming every question has an answer.

In [32]:
test_questions = [
    "How many days do employees have to submit travel expenses?",
    "Can employees work from home without manager approval?"
]

for query in test_questions:
    print("\n" + "=" * 60)
    print("Question:", query)

    response = openai_client.embeddings.create(
        model=os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"],
        input=query
    )

    vector_query = VectorizedQuery(
        vector=response.data[0].embedding,
        k_nearest_neighbors=3,
        fields="content_vector"
    )

    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["title", "source", "content"],
        top=3
    )

    for result in results:
        print("\nPolicy:", result.get("title"))
        print("Source:", result.get("source"))
        print("Content:", result.get("content"))


Question: How many days do employees have to submit travel expenses?

Policy: Business Travel Policy
Source: TRV-002
Content: Business travel must be approved by the reporting manager
        before booking. Economy class should be used for domestic
        flights. Employees must submit travel expenses with receipts
        within seven working days after completing the trip.

Policy: Work From Home Policy
Source: HR-001
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

Policy: Learning and Development Policy
Source: CDP-004
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Employees attending sponsored certification programs may
        be required to share their learning with thei

### Annotation — Define the Azure AI Search RAG answer function
`ask_policy_assistant(question)` combines retrieval and generation. It embeds the question, retrieves relevant chunks, handles an empty result set, and formats the retrieved content into a context string. It then sends the question and context to the chat deployment with instructions to avoid unsupported claims. Returning both the answer and source records supports inspection and citation. This function belongs to the Azure AI Search path; the later Pinecone function is a separate implementation.

In [33]:
from openai import OpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
import os

# Azure OpenAI client
openai_client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/") + "/"
)

# Azure AI Search client
search_client = SearchClient(
    endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
    index_name="policypilot-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_KEY"])
)

EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBED_DEPLOYMENT"]
CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]


def ask_policy_assistant(question):
    # 1. Convert the user's question into an embedding
    embedding_response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=question
    )
    query_vector = embedding_response.data[0].embedding

    # 2. Retrieve relevant policy chunks
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"
    )

    search_results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["title", "source", "department", "content"],
        top=3
    )

    retrieved_chunks = list(search_results)

    if not retrieved_chunks:
        return "No matching policy documents were found.", []

    # 3. Prepare the context for the language model
    context = "\n\n".join(
        f"Policy: {item.get('title', 'Unknown')}\n"
        f"Source: {item.get('source', 'Unknown')}\n"
        f"Content: {item.get('content', '')}"
        for item in retrieved_chunks
    )

    # 4. Ask Azure OpenAI to answer using the retrieved context
    completion = openai_client.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a company policy assistant. Answer using only "
                    "the supplied policy context. If the context does not "
                    "contain the answer, say that the policy information "
                    "provided is insufficient. Do not invent policy rules."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Policy context:\n{context}\n\n"
                    f"Employee question: {question}"
                )
            }
        ],
        temperature=0
    )

    answer = completion.choices[0].message.content
    return answer, retrieved_chunks

### Annotation — Execute one end-to-end Azure AI Search RAG test
This calls `ask_policy_assistant()` for a phishing-email question and prints both the answer and retrieved source details. It validates the complete retrieve-then-generate flow, not just the vector search component.

In [34]:
question = "What should employees do if they receive a phishing email?"

answer, sources = ask_policy_assistant(question)

print("QUESTION:")
print(question)

print("\nRAG ANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for source in sources:
    print(
        f"- {source.get('title')} "
        f"({source.get('source')}, "
        f"Department: {source.get('department')})"
    )

QUESTION:
What should employees do if they receive a phishing email?

RAG ANSWER:
Employees must report suspected phishing emails immediately to the IT security team, as stated in the Information Security Policy (Source: SEC-003).

RETRIEVED SOURCES:
- Information Security Policy (SEC-003, Department: IT)
- Work From Home Policy (HR-001, Department: HR)
- Learning and Development Policy (CDP-004, Department: Legal)


### Annotation — Test the Azure AI Search RAG function across topics
The question loop exercises the same RAG function on travel, work-from-home, authentication, and purchasing questions. This is a small functional test set. For each answer, check both the generated wording and the retrieved sources; plausible wording is not sufficient if the evidence is wrong.

In [35]:
questions = [
    "How many days do employees have to submit travel expenses?",
    "Can employees work from home without manager approval?",
    "What authentication method must employees use?",
    "What is the policy for purchasing items for the company?"
]

for question in questions:
    answer, sources = ask_policy_assistant(question)

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", answer)
    print("SOURCES:", ", ".join(
        str(source.get("title", "Unknown")) for source in sources
    ))


QUESTION: How many days do employees have to submit travel expenses?
ANSWER: Employees must submit travel expenses with receipts within seven working days after completing the trip, as per the Business Travel Policy (TRV-002).
SOURCES: Business Travel Policy, Work From Home Policy, Learning and Development Policy

QUESTION: Can employees work from home without manager approval?
ANSWER: The policy information provided is insufficient. The Work From Home Policy (HR-001) states that employees may work from home up to two days per week with manager approval, but it does not address whether working from home without manager approval is allowed or under what circumstances.
SOURCES: Work From Home Policy, Learning and Development Policy, Business Travel Policy

QUESTION: What authentication method must employees use?
ANSWER: The policy information provided specifies that employees must use multifactor authentication for company systems, as stated in the Information Security Policy (Source: S

### Annotation — Define manual RAG evaluation cases
The evaluation records pair each question with an expected outcome and a category. Answerable questions should be supported by policy text; out-of-scope questions should trigger an explicit uncertainty response. The cell inspects retrieved chunks and expected behaviour, but it does not automatically calculate a numeric quality score.

In [36]:

# Day 19: RAG Answer Quality Evaluation

evaluation_questions = [
    {
        "question": "What should employees do if they receive a phishing email?",
        "expected": "The answer should be supported by the Information Security Policy.",
        "type": "Answerable"
    },
    {
        "question": "What approval is required before booking business travel?",
        "expected": "The answer should be supported by the Business Travel Policy.",
        "type": "Answerable"
    },
    {
        "question": "What is the company's annual bonus percentage?",
        "expected": "If the policy corpus does not specify this, say so.",
        "type": "Out of scope"
    },
    {
        "question": "What is the CEO's personal phone number?",
        "expected": "Do not invent or disclose a number from unsupported sources.",
        "type": "Out of scope"
    },
    {
        "question": "How many days of paid parental leave are provided?",
        "expected": "Answer only if the retrieved policy explicitly specifies it.",
        "type": "Check coverage"
    }
]

for item in evaluation_questions:
    print("\n" + "=" * 75)
    print("TEST TYPE:", item["type"])
    print("QUESTION:", item["question"])
    print("EXPECTED:", item["expected"])

    retrieved_docs = faiss_store.similarity_search(
        item["question"], k=3
    )

    print("\nRETRIEVED POLICY CHUNKS:")
    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\n{rank}. Source: {doc.metadata.get('source')}")
        print("Title:", doc.metadata.get("title"))
        print("Content:", doc.page_content[:700])



TEST TYPE: Answerable
QUESTION: What should employees do if they receive a phishing email?
EXPECTED: The answer should be supported by the Information Security Policy.

RETRIEVED POLICY CHUNKS:

1. Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

2. Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

3. Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Em

### Annotation — Duplicate evaluation cell
This repeats the evaluation setup and checks from the preceding cell. It can be removed or repurposed for a distinct evaluation method to avoid confusing learners with duplicated output.

In [37]:

# Day 19: RAG Answer Quality Evaluation

evaluation_questions = [
    {
        "question": "What should employees do if they receive a phishing email?",
        "expected": "The answer should be supported by the Information Security Policy.",
        "type": "Answerable"
    },
    {
        "question": "What approval is required before booking business travel?",
        "expected": "The answer should be supported by the Business Travel Policy.",
        "type": "Answerable"
    },
    {
        "question": "What is the company's annual bonus percentage?",
        "expected": "If the policy corpus does not specify this, say so.",
        "type": "Out of scope"
    },
    {
        "question": "What is the CEO's personal phone number?",
        "expected": "Do not invent or disclose a number from unsupported sources.",
        "type": "Out of scope"
    },
    {
        "question": "How many days of paid parental leave are provided?",
        "expected": "Answer only if the retrieved policy explicitly specifies it.",
        "type": "Check coverage"
    }
]

for item in evaluation_questions:
    print("\n" + "=" * 75)
    print("TEST TYPE:", item["type"])
    print("QUESTION:", item["question"])
    print("EXPECTED:", item["expected"])

    retrieved_docs = faiss_store.similarity_search(
        item["question"], k=3
    )

    print("\nRETRIEVED POLICY CHUNKS:")
    for rank, doc in enumerate(retrieved_docs, start=1):
        print(f"\n{rank}. Source: {doc.metadata.get('source')}")
        print("Title:", doc.metadata.get("title"))
        print("Content:", doc.page_content[:700])



TEST TYPE: Answerable
QUESTION: What should employees do if they receive a phishing email?
EXPECTED: The answer should be supported by the Information Security Policy.

RETRIEVED POLICY CHUNKS:

1. Source: SEC-003
Title: Information Security Policy
Content: Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
        to the IT security team.

2. Source: HR-001
Title: Work From Home Policy
Content: Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        information must only be accessed through approved systems.

3. Source: CDP-004
Title: Learning and Development Policy
Content: Employees may request approval for job-related training.
        Training budgets are subject to departmental approval.
        Em

### Annotation — Install Pinecone integration
This installs the Pinecone SDK and LangChain adapter. Pinecone provides a hosted vector database, unlike the local FAISS index. The integration lets the notebook use a similar LangChain retrieval interface across different vector-store backends.

In [38]:

%pip install -q pinecone langchain-pinecone


### Annotation — Create or verify the Pinecone index
The API key is read from Colab Secrets rather than printed or hardcoded. The index name is `policypilot`; its dimension is derived from the embedding model's output, and cosine similarity is selected as the distance metric. The serverless specification chooses AWS and a region. Confirm the index region and dimensions match your Pinecone account and embedding deployment before ingestion.

In [39]:

import os
from google.colab import userdata

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "policypilot"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=len(emb.embed_query("probe")),
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Pinecone index is ready:", index_name)


Pinecone index is ready: policypilot


### Annotation — Ingest policy chunks and test Pinecone retrieval
`PineconeVectorStore.from_documents()` embeds the supplied `docs` and writes vectors plus metadata to the named index. The subsequent test questions call `similarity_search()` to check retrieval. Be aware that rerunning ingestion may add duplicate records if stable IDs or an explicit upsert strategy are not used; use stable chunk IDs for repeatable production ingestion.

In [40]:

pine_store = PineconeVectorStore.from_documents(
    docs,
    embedding=emb,
    index_name="policypilot"
)

test_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?"
]

for question in test_questions:
    print("\nQUESTION:", question)

    results = pine_store.similarity_search(question, k=3)

    for rank, doc in enumerate(results, start=1):
        print(
            f"{rank}. Source: {doc.metadata.get('source')} | "
            f"Policy: {doc.metadata.get('title')}"
        )
        print(doc.page_content[:400])



QUESTION: What should I do if my work laptop is stolen?
1. Source: SEC-003 | Policy: Information Security Policy
3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.
 
4. Cloud storage. Client or company data must never be stored on personal cloud accounts such as personal Google Drive or Dropbox.
2. Source: SEC-003 | Policy: Information Security Policy
Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corporate applications and VPN access.
 
2. Devices. All laptops are encrypted by default. USB storage devices are disabled by default; exceptions require written approval from the IT Security team.
3. Source: SEC-003 | Policy: Information Security Policy
Employe

### Annotation — Compare FAISS and Pinecone retrieval
This sends the same questions to both vector stores and compares the returned source chunks. The exercise demonstrates that the retrieval API can stay similar while the storage backend changes. The comparison is qualitative unless you calculate metrics such as Hit@K or MRR, and it is only fair when both stores contain the same corpus and use compatible embeddings.

In [41]:

comparison_questions = [
    "What should I do if my work laptop is stolen?",
    "Can I book business class for an eight-hour flight?",
    "How long are client engagement records kept?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for question in comparison_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    print("\nFAISS:")
    for doc in faiss_store.similarity_search(question, k=2):
        print(doc.metadata.get("source"), "|", doc.page_content[:200])

    print("\nPINECONE:")
    for doc in pine_store.similarity_search(question, k=2):
        print(doc.metadata.get("source"), "|", doc.page_content[:200])



QUESTION: What should I do if my work laptop is stolen?

FAISS:
SEC-003 | Employees must use multifactor authentication for company
        systems. Passwords must never be shared. Suspected phishing
        emails and security incidents must be reported immediately
       
HR-001 | Employees may work from home up to two days per week with
        manager approval. Employees must remain available during
        core business hours from 10 AM to 4 PM. Confidential company
        

PINECONE:
SEC-003 | 3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not a
SEC-003 | Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corpor

QUESTION: Can I book business class for an eight-hour flight?

FAISS:
TRV-002 | 

### Annotation — Verify Pinecone retrieval before generation
This tests several policy questions against Pinecone and prints source metadata and chunk content. It is a checkpoint to ensure the index contains useful policy material before adding the LLM generation stage. If a question is outside the corpus, inspect the retrieved content and make sure the assistant does not infer a policy that is not present.

In [42]:

# Verify that Pinecone contains and retrieves policy chunks

test_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for question in test_questions:
    print("\n" + "=" * 75)
    print("QUESTION:", question)

    results = pine_store.similarity_search(question, k=3)

    if not results:
        print("No matching policy chunks retrieved.")
        continue

    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("Source:", doc.metadata.get("source", "Not available"))
        print("Title:", doc.metadata.get("title", "Not available"))
        print("Policy text:", doc.page_content[:500])



QUESTION: What should I do if my work laptop is stolen?

Result 1
Source: SEC-003
Title: Information Security Policy
Policy text: 3. Lost or stolen devices. Any lost or stolen device must be reported to the Security Operations Center (SOC) within 1 hour of discovery, using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.
 
4. Cloud storage. Client or company data must never be stored on personal cloud accounts such as personal Google Drive or Dropbox.

Result 2
Source: SEC-003
Title: Information Security Policy
Policy text: Meridian Global Advisory - Information Security Policy (SEC-003)
 
1. Passwords and access. Passwords must be at least 14 characters long. Multi-factor authentication (MFA) is mandatory for all corporate applications and VPN access.
 
2. Devices. All laptops are encrypted by default. USB storage devices are disabled by default; exceptions require written approval from the IT Security team.

Result 3
Source: SEC-003
Titl

### Annotation — Install final client dependencies
This installs `python-dotenv` and the OpenAI SDK for the final `.env`-based implementation. These packages were introduced earlier, so this is a self-contained setup cell rather than a new pipeline stage.

In [43]:
%pip install -q python-dotenv openai

### Annotation — Earlier AzureOpenAI client attempt
This cell reads `.env` and prepares an Azure-specific client using the traditional API-version configuration. It records an earlier approach in the notebook; your final working implementation uses the OpenAI-compatible client with an Azure `/openai/v1/` base URL in Cell 45. Keep this cell only as a comparison/debugging example and avoid executing conflicting client initialization unnecessarily.

In [44]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

# Load variables from your existing .env file
load_dotenv(".env", override=False)

# Read Azure OpenAI settings from .env
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# Find the deployment name using common .env variable names
chat_deployment = (
    os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
)

# Validate configuration without displaying secrets
missing = [
    name for name, value in {
        "AZURE_OPENAI_API_KEY": api_key,
        "AZURE_OPENAI_ENDPOINT": endpoint,
        "AZURE_OPENAI_API_VERSION": api_version,
        "Azure OpenAI deployment name": chat_deployment,
    }.items()
    if not value
]

if missing:
    print("Missing configuration:", ", ".join(missing))
    print("Check your .env variable names and file location.")
else:
    chat_client = AzureOpenAI(
        api_key=api_key,
        azure_endpoint=endpoint,
        api_version=api_version
    )
    print("Azure OpenAI client initialized successfully.")
    print("Deployment configuration found.")

Azure OpenAI client initialized successfully.
Deployment configuration found.


### Annotation — Confirm the `.env` file is present
`os.getcwd()` reports the notebook's current working directory, and `os.path.isfile('.env')` checks whether `.env` exists there. This helps diagnose configuration loading failures in Colab. If the file is missing, upload or create it securely; do not commit it to the repository.

In [45]:
import os

print("Current directory:", os.getcwd())
print(".env exists:", os.path.isfile(".env"))

Current directory: /content
.env exists: True


### Annotation — Final working PolicyPilot RAG pipeline
This cell loads `.env`, validates the required endpoint/key/deployment settings, constructs the Azure OpenAI v1 base URL, and initializes the `OpenAI` client. `answer_from_pinecone()` retrieves the top `k` chunks from Pinecone, formats source/title/content into context, and asks the chat deployment to answer only from that context. `temperature=0` reduces response variability. The loop runs each test question and reports runtime errors. Ensure `pine_store` and `test_questions` have already been created in earlier cells. Do not print or commit secrets; rotate any credentials that were exposed in notebook screenshots.

In [46]:

%pip install -q python-dotenv openai

import os
from dotenv import load_dotenv
from openai import OpenAI

# ==========================================
# 1. Load your existing .env file
# ==========================================
load_dotenv(".env", override=True)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

# Validate required settings
missing = [
    name for name, value in {
        "AZURE_OPENAI_ENDPOINT": endpoint,
        "AZURE_OPENAI_API_KEY": api_key,
        "AZURE_OPENAI_CHAT_DEPLOYMENT": chat_deployment,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        "Missing .env variables: " + ", ".join(missing)
    )

# ==========================================
# 2. Configure Azure OpenAI v1 client
# ==========================================
if endpoint.endswith("/openai/v1"):
    base_url = endpoint + "/"
elif endpoint.endswith("/openai"):
    base_url = endpoint + "/v1/"
else:
    base_url = endpoint + "/openai/v1/"

chat_client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

print("Azure OpenAI v1 client initialized.")
print("Chat deployment:", chat_deployment)
print("API version setting found:", bool(api_version))

# ==========================================
# 3. PolicyPilot RAG answer function
# Requires pine_store from your Pinecone setup
# ==========================================
def answer_from_pinecone(question, k=3):
    retrieved_docs = pine_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n"
        f"Policy: {doc.metadata.get('title', 'Unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = f"""
Answer using only the supplied policy excerpts.

Rules:
- Do not invent policy rules or facts.
- If there is insufficient evidence, say:
  "I couldn't find enough information in the available policies
  to answer this question."
- Cite the supporting policy source when possible.
- Related text alone does not prove that a question is answerable.

POLICY EXCERPTS:
{context if context else "No policy excerpts retrieved."}

QUESTION:
{question}

Give a concise answer, cite the supporting source,
or explain that the information was not found.
"""

    response = chat_client.chat.completions.create(
        model=chat_deployment,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are PolicyPilot, an evidence-based "
                    "policy assistant."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


# ==========================================
# 4. Test the RAG pipeline
# Requires test_questions from your earlier cell
# ==========================================
for question in test_questions:
    print("\n" + "=" * 75)
    print("QUESTION:", question)

    try:
        print(answer_from_pinecone(question))
    except Exception as e:
        print("Error type:", type(e).__name__)
        print("Error details:", str(e))
        break


Azure OpenAI v1 client initialized.
Chat deployment: gpt-4o-1
API version setting found: True

QUESTION: What should I do if my work laptop is stolen?
Report the stolen laptop to the Security Operations Center (SOC) within 1 hour of discovery using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.  
(Source: SEC-003, Information Security Policy, Section 3)

QUESTION: What is the hotel limit for a Tier-1 city?
The hotel limit for a Tier-1 city is 180 USD per night.  
(Source: TRV-002, Travel & Expense Policy, Section 2)

QUESTION: How long are client engagement records retained?
Client engagement records are retained for 7 years after engagement closure. (Source: CDP-004, Section 4)

QUESTION: What is the company's annual bonus percentage?
"I couldn't find enough information in the available policies to answer this question."

QUESTION: What is the company's policy on cryptocurrency trading?
"I couldn't find enough information in the availabl

### Annotation — Final PolicyPilot evaluation set
This cell defines five representative test questions and runs each through Pinecone retrieval and the answer-generation function. The first three should be answerable only if the relevant policy documents are present in the indexed corpus; the bonus and cryptocurrency questions are intended to test missing-information handling. Inspect the retrieved sources as well as the answer. A convincing answer is not a correct answer unless the source evidence supports it.

In [47]:

evaluation_questions = [
    "What should I do if my work laptop is stolen?",
    "What is the hotel limit for a Tier-1 city?",
    "How long are client engagement records retained?",
    "What is the company's annual bonus percentage?",
    "What is the company's policy on cryptocurrency trading?"
]

for i, question in enumerate(evaluation_questions, start=1):
    print(f"\n{'=' * 75}")
    print(f"TEST CASE {i}: {question}")

    retrieved_docs = pine_store.similarity_search(question, k=3)

    print("\nRetrieved sources:")
    for doc in retrieved_docs:
        print("-", doc.metadata.get("source", "Unknown"))

    print("\nPolicyPilot answer:")
    print(answer_from_pinecone(question))



TEST CASE 1: What should I do if my work laptop is stolen?

Retrieved sources:
- SEC-003
- SEC-003
- SEC-003

PolicyPilot answer:
Report the stolen laptop to the Security Operations Center (SOC) within 1 hour of discovery using the SOC hotline or the Report an Incident form. Do not attempt to recover the device yourself.  
(Source: SEC-003, Information Security Policy, Section 3)

TEST CASE 2: What is the hotel limit for a Tier-1 city?

Retrieved sources:
- TRV-002
- TRV-002
- TRV-002

PolicyPilot answer:
The hotel limit for a Tier-1 city is 180 USD per night.  
(Source: TRV-002, Travel & Expense Policy, Section 2. Accommodation)

TEST CASE 3: How long are client engagement records retained?

Retrieved sources:
- CDP-004
- CDP-004
- CDP-004

PolicyPilot answer:
Client engagement records are retained for 7 years after engagement closure. (Source: CDP-004, Section 4)

TEST CASE 4: What is the company's annual bonus percentage?

Retrieved sources:
- HR-001
- CDP-004
- SEC-003

PolicyPilo